# DEEP LEARNING HOME LENDING CAPSTONE

In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import os

# --- Configuration and Setup ---
DATA_FILE = 'loan_data.csv'

# Set display options for better data visibility
pd.set_option('display.max_columns', 50)
sns.set_style('whitegrid')

print("Starting Loan Data Analysis project (Home Credit Dataset detected)...")


try:
    df = pd.read_csv(DATA_FILE)
    print(f"\nSuccessfully loaded {DATA_FILE}. Shape: {df.shape}")
except FileNotFoundError:
    print(f"\nERROR: File '{DATA_FILE}' not found. Please ensure the CSV file is in the same directory.")
    df = pd.DataFrame() 

# Correctly identifying the target column based on the output
TARGET_COL = 'TARGET'

Starting Loan Data Analysis project (Home Credit Dataset detected)...

Successfully loaded loan_data.csv. Shape: (307511, 122)


## EDA & Categorical Encoding

In [26]:
if not df.empty:
    print("\n--- Initial Data Check & Cleaning ---")
    # Drop the ID column as it's not useful for modeling
    if 'SK_ID_CURR' in df.columns:
        df = df.drop('SK_ID_CURR', axis=1)
        print("Dropped 'SK_ID_CURR' column.")
        
    print(df.head())
    print(f"\nNew Shape: {df.shape}")
    print("\n--- Column Info and Missing Values ---")
    df.info(verbose=False, show_counts=True)

    
    # Check the balance of the target variable ('TARGET')
    if TARGET_COL in df.columns:
        target_counts = df[TARGET_COL].value_counts(normalize=True) * 100
        print(f"\n--- Target Variable Distribution ('{TARGET_COL}') ---")
        print(f"Loan Repaid (0): {target_counts.get(0, 0):.2f}%")
        print(f"Loan Defaulted (1): {target_counts.get(1, 0):.2f}%")
        print("The dataset is highly imbalanced (approximately 92/8 split).")
    else:
        print(f"ERROR: The target column '{TARGET_COL}' was not found after initial checks.")


    # Feature Transformation (One-Hot Encoding all Categorical Features)
    object_cols = df.select_dtypes('object').columns.tolist()
    
    print(f"\n--- 3. Feature Transformation (Task 1: One-Hot Encoding) ---")
    print(f"Identified {len(object_cols)} categorical features for OHE: {object_cols}")
    
    # Perform One-Hot Encoding, dropping the first category to avoid multicollinearity.
    df = pd.get_dummies(df, columns=object_cols, drop_first=True)
    
    # Check the new shape and columns
    print(f"New DataFrame shape after encoding: {df.shape}")
    print(f"Number of new features: {df.shape[1] - (len(df.columns) - len(object_cols))}")
    
    # Calculate the percentage of missing values per column
    missing_percent = (df.isnull().sum() / len(df)) * 100
    missing_df = pd.DataFrame({'Missing Percent': missing_percent})
    missing_df = missing_df[missing_df['Missing Percent'] > 0].sort_values(by='Missing Percent', ascending=False)
    
    print(f"\n{len(missing_df)} columns have missing values. Top 5 missing features before imputation:")
    print(missing_df.head(5))



--- Initial Data Check & Cleaning ---
Dropped 'SK_ID_CURR' column.
   TARGET NAME_CONTRACT_TYPE CODE_GENDER FLAG_OWN_CAR FLAG_OWN_REALTY  \
0       1         Cash loans           M            N               Y   
1       0         Cash loans           F            N               N   
2       0    Revolving loans           M            Y               Y   
3       0         Cash loans           F            N               Y   
4       0         Cash loans           M            N               Y   

   CNT_CHILDREN  AMT_INCOME_TOTAL  AMT_CREDIT  AMT_ANNUITY  AMT_GOODS_PRICE  \
0             0          202500.0    406597.5      24700.5         351000.0   
1             0          270000.0   1293502.5      35698.5        1129500.0   
2             0           67500.0    135000.0       6750.0         135000.0   
3             0          135000.0    312682.5      29686.5         297000.0   
4             0          121500.0    513000.0      21865.5         513000.0   

  NAME_TYPE_SUITE 

## Missing Value Imputation

In [27]:
# Impute missing numerical values with the median of the column
# Impute all remaining columns (excluding TARGET) with their median
for col in df.columns:
    if col != TARGET_COL and df[col].isnull().any():
        # Use the median for numerical imputation
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)

print(f"All numerical features (excluding '{TARGET_COL}') have been imputed with their column median.")

# Re-check missing values
missing_after_imputation = df.isnull().sum().sum()
if missing_after_imputation == 0:
    print("Successfully imputed all missing values. DataFrame is complete.")
else:
        print(f"WARNING: {missing_after_imputation} missing values remain (This should ideally be 0).")

All numerical features (excluding 'TARGET') have been imputed with their column median.
Successfully imputed all missing values. DataFrame is complete.


## Feature Correlation Check

In [28]:
# Calculate the absolute correlation matrix
corr_matrix = df.corr().abs()

# Select upper triangle of correlation matrix
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Define a strict threshold for highly correlated features
THRESHOLD = 0.98 

# Find features with correlation greater than the threshold
to_drop = [column for column in upper.columns if any(upper[column] > THRESHOLD)]

# Drop features 
df = df.drop(columns=to_drop)

print(f"Dropped {len(to_drop)} highly correlated features (Correlation > {THRESHOLD}):")
print(to_drop)
print(f"New DataFrame shape after dropping correlated features: {df.shape}")

Dropped 23 highly correlated features (Correlation > 0.98):
['AMT_GOODS_PRICE', 'FLAG_EMP_PHONE', 'YEARS_BUILD_MODE', 'ELEVATORS_MODE', 'FLOORSMAX_MODE', 'FLOORSMIN_MODE', 'APARTMENTS_MEDI', 'BASEMENTAREA_MEDI', 'YEARS_BEGINEXPLUATATION_MEDI', 'YEARS_BUILD_MEDI', 'COMMONAREA_MEDI', 'ELEVATORS_MEDI', 'ENTRANCES_MEDI', 'FLOORSMAX_MEDI', 'FLOORSMIN_MEDI', 'LANDAREA_MEDI', 'LIVINGAPARTMENTS_MEDI', 'LIVINGAREA_MEDI', 'NONLIVINGAPARTMENTS_MEDI', 'NONLIVINGAREA_MEDI', 'OBS_60_CNT_SOCIAL_CIRCLE', 'NAME_INCOME_TYPE_Pensioner', 'ORGANIZATION_TYPE_XNA']
New DataFrame shape after dropping correlated features: (307511, 206)


## Modeling and Evaluation

In [30]:
# Define features (X) and target (y)
X = df.drop(TARGET_COL, axis=1)
y = df[TARGET_COL]

# Split the data into training and testing sets (80/20 split, stratified for imbalance)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

# Standard Scaling
scaler = StandardScaler()

# Fit scaler on training data and transform both training and testing data
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features scaled using StandardScaler (fit only on X_train).")


# --- Calculate Class Weights for Imbalanced Data ---
from sklearn.utils import class_weight 
weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
# Convert weights array to dictionary format {class_label: weight}
class_weights_dict = dict(zip(np.unique(y_train), weights))

print(f"\nCalculated Class Weights: {class_weights_dict}")
print(f"The minority class (1, Default) is now weighted {class_weights_dict[1]:.2f}x higher than the majority class (0, Paid).")


# --- Deep Learning Model ---
input_dim = X_train_scaled.shape[1]

# Defining the model architecture (A simple, deep network)
model = Sequential([
    # Input layer and first hidden layer
    Dense(64, activation='relu', input_shape=(input_dim,)),
    Dropout(0.3), # Reverted dropout to 0.3
    
    # Second hidden layer
    Dense(32, activation='relu'),
    Dropout(0.3), # Reverted dropout to 0.3
    
    # Output layer (Sigmoid for binary classification)
    Dense(1, activation='sigmoid')
])

# Compile the model
model.compile(optimizer=Adam(learning_rate=0.001), 
                loss='binary_crossentropy', 
                metrics=['accuracy', 'AUC']) 

model.summary()

# Training the model - NOW INCLUDING CLASS WEIGHTS
from tensorflow.keras.callbacks import EarlyStopping

# Define Early Stopping to monitor validation AUC and stop if it doesn't improve
early_stopping = EarlyStopping(
    monitor='val_AUC',  
    patience=5,         
    mode='max',         
    restore_best_weights=True
)
print("\nStarting Model Retraining with Class Weights (15 epochs max, with Early Stopping on val_AUC with patience 5)...")

history = model.fit(
    X_train_scaled, 
    y_train, 
    epochs=15, # DECREASED from 50 to 15
    batch_size=256, 
    validation_data=(X_test_scaled, y_test),
    class_weight=class_weights_dict, 
    callbacks=[early_stopping], # Added callback
    verbose=1
)

print("\nModel retraining complete.")

# --- Model Evaluation ---
print("\n--- 8. Model Evaluation (Default Threshold 0.5) ---")

# Predict probabilities on the test set
y_pred_proba = model.predict(X_test_scaled).flatten()

# Calculate AUC score
auc_score = roc_auc_score(y_test, y_pred_proba)
print(f"\nArea Under the ROC Curve (AUC): {auc_score:.4f}")

# Convert probabilities to binary predictions using a default threshold of 0.5
y_pred = (y_pred_proba > 0.5).astype(int)

# Classification Report
print("\nClassification Report (Threshold 0.5):")
print(classification_report(y_test, y_pred))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix (True Negatives, False Positives, False Negatives, True Positives):")
print(cm)


# --- Optimal Threshold Search ---
print("\n" + "="*50)
print("--- 9. Optimal Threshold Search to Maximize F1-Score (Class 1) ---") # Updated title
print("="*50)

# Define the range of thresholds to test
# Testing from 0.05 up to 0.80
thresholds = np.arange(0.05, 0.81, 0.01) # Increased granularity for a better F1 search

best_f1 = 0
best_threshold = 0.5
best_report = ""

# Loop through each threshold and evaluate the performance
print("Testing various thresholds for improved Class 1 F1-Score:")
for t in thresholds:
    # Convert probabilities to binary predictions using the current threshold 't'
    y_pred_t = (y_pred_proba > t).astype(int)
    
    # Calculate the classification report
    report = classification_report(y_test, y_pred_t, output_dict=True, zero_division=0)
    
    # Extract the F1-Score for the minority class (Class 1)
    current_f1 = report['1']['f1-score']
    
    # Check if this threshold gives better F1-Score
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_threshold = t
        best_report = classification_report(y_test, y_pred_t, zero_division=0)
    if t % 0.05 < 0.001 or t >= 0.80: 
        print(f"Threshold {t:.2f} -> F1-Score (Class 1): {current_f1:.4f}")

# Print the results for the best threshold found
print("\n" + "="*50)
print(f"Optimal Threshold Found: {best_threshold:.2f}")
print(f"Max F1-Score (Class 1) Achieved: {best_f1:.4f}")
print("Optimal Classification Report:")
print(best_report)
print("="*50)

X_train shape: (246008, 205), y_train shape: (246008,)
X_test shape: (61503, 205), y_test shape: (61503,)
Features scaled using StandardScaler (fit only on X_train).

Calculated Class Weights: {0: 0.5439092983356032, 1: 6.193554884189325}
The minority class (1, Default) is now weighted 6.19x higher than the majority class (0, Paid).


/opt/miniconda3/envs/tf-mac-metal/lib/python3.11/site-packages/keras/src/layers/core/dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_12 (Dense)                │ (None, 64)             │        13,184 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,297 (59.75 KB)

 Trainable params: 15,297 (59.75 KB)

 Non-trainable params: 0 (0.00 B)


Starting Model Retraining with Class Weights (15 epochs max, with Early Stopping on val_AUC with patience 5)...
Epoch 1/15
961/961 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - AUC: 0.6573 - accuracy: 0.6167 - loss: 0.7040 - val_AUC: 0.7432 - val_accuracy: 0.6676 - val_loss: 0.6175
Epoch 2/15
961/961 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - AUC: 0.7195 - accuracy: 0.6624 - loss: 0.6196 - val_AUC: 0.7440 - val_accuracy: 0.6768 - val_loss: 0.6089
Epoch 3/15
961/961 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - AUC: 0.7299 - accuracy: 0.6735 - loss: 0.6109 - val_AUC: 0.7435 - val_accuracy: 0.6932 - val_loss: 0.5969
Epoch 4/15
961/961 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - AUC: 0.7330 - accuracy: 0.6791 - loss: 0.6080 - val_AUC: 0.7441 - val_accuracy: 0.6807 - val_loss: 0.6033
Epoch 5/15
961/961 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - AUC: 0.7345 - accuracy: 0.6814 - loss: 0.6067 - val_AUC: 0.7436 - val_accuracy: 0.6755 - val_loss: 0.6102
Epoch 6/15
961/961 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - AUC: 0.7339 - accurac